# spharmgrid — ERA5 850-hPa spherical harmonic explorer

Interactive comparison of ERA5 850-hPa wind before and after vector spherical-harmonic filtering/regridding. The same notebook works in JupyterLab and with `panel serve`.

In [ ]:
from functools import lru_cache
from pathlib import Path
import cartopy.crs as ccrs
import geoviews as gv
import holoviews as hv
import numpy as np
import panel as pn
import pooch
import xarray as xr
import spharmgrid as sg

pn.extension(); hv.extension("bokeh")
TAG="v0.2.0-data"; BASE=f"https://github.com/mwyau/PyStormTracker-Data/releases/download/{TAG}/"
UV="era5_uv850_2025-2026_djf_2.5x2.5.nc"; VO="era5_vo850_2025-2026_djf_2.5x2.5.nc"
REG={UV:"sha256:43cbc346a52c5230ac34eb22c7a640800fbffad40da4058686c8042a76bc5965",VO:"sha256:46ce78cd3b065d3777c2d628cdc2311d68a9fcb4d3a3b9948db7c7376ae7a6aa"}
DATA=pooch.create(path=Path(pooch.os_cache("spharmgrid"))/"interactive-v0.2.0-data",base_url=BASE,registry=REG)

def fetch(name): return Path(DATA.fetch(name,progressbar=False))
def pressure_hpa(c):
    x=float(c.values[0]); u=str(c.attrs.get("units","")).strip().lower()
    if u in {"pa","pascal","pascals"}: return x/100
    if u in {"hpa","hectopascal","hectopascals","mbar","millibar","millibars"} or (not u and np.isclose(x,850)): return x
    if not u and np.isclose(x,85000): return x/100
    raise ValueError(f"Unsupported pressure units {c.attrs.get('units')!r}")

with xr.open_dataset(fetch(UV),engine="h5netcdf") as r:
    if {"u","v"}-set(r.data_vars): raise ValueError("Pinned ERA5 file must contain u and v")
    if r.sizes.get("pressure_level")!=1 or not np.isclose(pressure_hpa(r.pressure_level),850): raise ValueError("Expected one 850-hPa level")
    ERA5=r[["u","v"]].isel(pressure_level=0,drop=True).load()
TIMES=ERA5.valid_time.values
source_grid=sg.detect_grid(ERA5.u.isel(valid_time=0))
if source_grid.kind!="cc": raise ValueError(f"Expected CC grid, found {source_grid.kind}")
def tmax(g): return min(g.nlat-2 if g.kind=="cc" else g.nlat-1,(g.nlon-1)//2)
LMAX=tmax(source_grid); ORDER="descending" if source_grid.latitude[0]>source_grid.latitude[-1] else "ascending"
gl_target=sg.gaussian_grid(source_grid.nlat-1,source_grid.nlon,lon0=float(source_grid.longitude[0]),latitude_order=ORDER)
if LMAX<42: raise ValueError(f"Pinned grid supports only T{LMAX}")

frame0=ERA5.isel(valid_time=0)
{"grid":frame0.sg.grid_type,"shape":ERA5.u.shape,"range":f"T0-{LMAX}","kinematics":list(frame0.sg.kinematics().data_vars),"potentials":list(frame0.sg.potentials().data_vars)}

## Processing

`ERA5 CC` filtering uses `sg.regrid_vector` with the source grid as the target, so `u` and `v` are processed together as a vector field. The optional GL mode performs the same spectral operation while synthesizing to a 72×144 Gauss–Legendre grid.

In [ ]:
DIAGS=("Wind","Relative vorticity","Divergence","Streamfunction","Velocity potential","Rotational wind","Divergent wind")
GRIDS=("ERA5 CC","Gauss–Legendre")
@lru_cache(maxsize=72)
def frame(i):
    x=ERA5.isel(valid_time=int(i)); return x.u,x.v
@lru_cache(maxsize=36)
def processed(i,lo,hi,taper,grid):
    target=source_grid if grid=="ERA5 CC" else gl_target
    if not 0<=lo<=hi<=min(LMAX,tmax(target)): raise ValueError(f"Invalid spectral range T{lo}-{hi}")
    u,v=frame(i); return sg.regrid_vector(u,v,target,lmin=lo,lmax=hi,taper=taper)
def state(u,v,d):
    if d=="Wind": return "wind",np.hypot(u,v),u,v,1.0,"Wind speed (m s⁻¹)"
    if d in {"Relative vorticity","Divergence"}:
        k=sg.kinematics(u,v); return ("scalar",k["vo"],None,None,1e5,"Relative vorticity (10⁻⁵ s⁻¹)") if d.startswith("Relative") else ("scalar",k["d"],None,None,1e5,"Divergence (10⁻⁵ s⁻¹)")
    if d in {"Streamfunction","Velocity potential"}:
        p=sg.potentials(u,v); return ("scalar",p["strf"],None,None,1e-6,"Streamfunction (10⁶ m² s⁻¹)") if d=="Streamfunction" else ("scalar",p["vp"],None,None,1e-6,"Velocity potential (10⁶ m² s⁻¹)")
    k=sg.kinematics(u,v)
    if d=="Rotational wind":
        w=sg.rotational_wind(k.vo,quantity="vorticity"); return "wind",np.hypot(w.u_rotational,w.v_rotational),w.u_rotational,w.v_rotational,1.0,"Rotational wind (m s⁻¹)"
    w=sg.divergent_wind(k.d,quantity="divergence"); return "wind",np.hypot(w.u_divergent,w.v_divergent),w.u_divergent,w.v_divergent,1.0,"Divergent wind (m s⁻¹)"
@lru_cache(maxsize=72)
def original_state(i,d): return state(*frame(i),d)
@lru_cache(maxsize=72)
def processed_state(i,lo,hi,taper,grid,d):
    w=processed(i,lo,hi,taper,grid); return state(w.u,w.v,d)

In [ ]:
CRS=ccrs.PlateCarree(); COAST=gv.feature.coastline(); STRIDE=4
def arr(x): return np.asarray(x.values,dtype=float)
def xy(field):
    f=field.transpose("latitude","longitude"); lon=(np.asarray(f.longitude)+180)%360-180; j=np.argsort(lon); return lon[j],np.asarray(f.latitude),arr(f)[:,j]
def uvxy(u,v):
    lon,lat,a=xy(u); _,_,b=xy(v); return lon,lat,a,b
def limits(a,b):
    x=np.concatenate((arr(a[1]).ravel()*a[4],arr(b[1]).ravel()*b[4])); x=x[np.isfinite(x)]
    if a[0]=="scalar": q=max(float(np.percentile(np.abs(x),99)),1e-12); return -q,q
    return 0,max(float(np.percentile(x,99)),1.0)
def opts(title): return dict(width=620,height=360,xlim=(-180,180),ylim=(-90,90),projection=CRS,title=title,tools=["hover","pan","wheel_zoom","reset"])
def draw(s,clim,title):
    kind,f,u,v,scale,label=s; lon,lat,z=xy(f); mesh=gv.QuadMesh((lon,lat,z*scale),kdims=["longitude","latitude"],vdims=[label],crs=CRS).opts(cmap="RdBu_r" if kind=="scalar" else "Viridis",colorbar=True,clim=clim,**opts(title)); coast=COAST.opts(line_color="#374151",line_width=.8,projection=CRS)
    if kind=="scalar": return mesh*coast
    lon,lat,a,b=uvxy(u,v); dl=lon[::STRIDE]; dy=lat[1:-1:STRIDE]; a=a[1:-1:STRIDE,::STRIDE]; b=b[1:-1:STRIDE,::STRIDE]
    vec=gv.VectorField((dl,dy,np.arctan2(b,a),np.hypot(a,b)),kdims=["longitude","latitude"],vdims=["angle","magnitude"],crs=CRS).opts(color="#111827",line_width=1,pivot="mid",projection=CRS,xlim=(-180,180),ylim=(-90,90))
    return mesh*vec*coast
def render(side,i,rng,taper_on,taper_value,grid,d):
    lo,hi=map(int,rng); taper=float(taper_value) if taper_on else None; a=original_state(int(i),d); b=processed_state(int(i),lo,hi,taper,grid,d); return draw(a if side=="original" else b,limits(a,b),f"{side.title()} · {d}")
def stamp(i): return np.datetime_as_string(TIMES[int(i)],unit="m").replace("T"," ")+" UTC"

## Interactive application

The spectral sliders use throttled values, so transforms run when a drag ends. A normal Panel layout is used here instead of `FastListTemplate`; embedding the full-page template in JupyterLab was the source of the oversized four-corner/fullscreen glyph.

In [ ]:
diag=pn.widgets.Select(name="Diagnostic",options=list(DIAGS),value="Wind")
player=pn.widgets.Player(name="Time",start=0,end=len(TIMES)-1,value=0,interval=1800,loop_policy="loop",show_value=False)
rng=pn.widgets.IntRangeSlider(name="Spectral degree range",start=0,end=LMAX,value=(0,42),step=1)
taper_on=pn.widgets.Checkbox(name="Taper enabled",value=False); taper_value=pn.widgets.FloatSlider(name="Taper endpoint response",start=.01,end=1,step=.01,value=.1,disabled=True)
grid=pn.widgets.Select(name="Output grid",options=list(GRIDS),value="ERA5 CC")
taper_on.param.watch(lambda e:setattr(taper_value,"disabled",not bool(e.new)),"value")
bind=dict(i=player.param.value,rng=rng.param.value_throttled,taper_on=taper_on.param.value,taper_value=taper_value.param.value_throttled,grid=grid.param.value,d=diag.param.value)
left=hv.DynamicMap(pn.bind(render,side="original",**bind),kdims=[]); right=hv.DynamicMap(pn.bind(render,side="processed",**bind),kdims=[])
summary=pn.pane.Markdown(pn.bind(lambda i,r,t,tv,g:f"**{stamp(i)}** · `T{r[0]}–{r[1]}` · "+(f"taper endpoint {tv:g}" if t else "hard selection")+f" · `{g}`",player.param.value,rng.param.value_throttled,taper_on.param.value,taper_value.param.value_throttled,grid.param.value))

vo_cache=None
def rel(a,b):
    a=arr(a); b=arr(b); den=np.sqrt(np.nanmean(a*a)); return float(np.sqrt(np.nanmean((a-b)**2))/den) if den else 0.0
def vrel(au,av,bu,bv): return max(rel(au,bu),rel(av,bv))
def checks(_):
    global vo_cache
    lo,hi=map(int,rng.value_throttled); t=float(taper_value.value_throttled) if taper_on.value else None; w=processed(player.value,lo,hi,t,grid.value); k=sg.kinematics(w.u,w.v); p=sg.potentials(w.u,w.v); restored=sg.wind(k.vo,k.d,source="vorticity_divergence"); h=sg.helmholtz(w.u,w.v); g=sg.gradient(p.vp); ig=sg.inverse_gradient(g.gradient_eastward,g.gradient_northward); rw=sg.rotational_wind(k.vo,quantity="vorticity"); dw=sg.divergent_wind(k.d,quantity="divergence"); vl=sg.vector_laplacian(w.u,w.v); vi=sg.inverse_vector_laplacian(vl.u,vl.v); lines=["### Consistency checks",f"wind reconstruction: `{vrel(w.u,w.v,restored.u,restored.v):.3e}`",f"Helmholtz sum: `{vrel(w.u,w.v,h.u_rotational+h.u_divergent,h.v_rotational+h.v_divergent):.3e}`",f"laplacian(strf) vs vo: `{rel(k.vo,sg.laplacian(p.strf)):.3e}`",f"laplacian(vp) vs d: `{rel(k.d,sg.laplacian(p.vp)):.3e}`",f"inverse_laplacian(vo) vs strf: `{rel(p.strf,sg.inverse_laplacian(k.vo)):.3e}`",f"inverse_gradient(gradient(vp)) vs vp: `{rel(p.vp,ig):.3e}`",f"rotational + divergent wind: `{vrel(w.u,w.v,rw.u_rotational+dw.u_divergent,rw.v_rotational+dw.v_divergent):.3e}`",f"vector Laplacian round trip: `{vrel(w.u,w.v,vi.u,vi.v):.3e}`"]
    try:
        if vo_cache is None:
            with xr.open_dataset(fetch(VO),engine="h5netcdf") as r: vo_cache=r.vo.isel(pressure_level=0,drop=True).load()
        ref=vo_cache.isel(valid_time=int(player.value)); calc=sg.vorticity(*frame(player.value)); a=arr(ref).ravel(); b=arr(calc).ravel(); ok=np.isfinite(a)&np.isfinite(b); lines.append(f"ERA5 vo correlation: `{np.corrcoef(a[ok],b[ok])[0,1]:.6f}`")
    except Exception as e: lines.append(f"External ERA5 vo comparison unavailable: `{e}`")
    check_out.object="\n\n".join(lines)
check_btn=pn.widgets.Button(name="Run checks for current frame",button_type="primary"); check_out=pn.pane.Markdown("Checks run on demand; the separate ERA5 `vo` file is downloaded only here."); check_btn.on_click(checks)
controls=pn.Column("### Controls",diag,player,pn.pane.Markdown(pn.bind(lambda i:f"**ERA5 timestamp:** `{stamp(i)}`",player.param.value)),rng,taper_on,taper_value,grid,width=300)
plots=pn.Row(pn.Column("### Original",left),pn.Column("### Processed",right))
main=pn.Column("# spharmgrid — ERA5 850-hPa spherical harmonic explorer","Compare original and processed ERA5 wind on the same timestamp.",plots,summary,pn.Card(check_btn,check_out,title="Consistency checks"),pn.pane.Markdown("**Data:** ERA5, Copernicus Climate Change Service / ECMWF; distributed through PyStormTracker-Data `v0.2.0-data`."))
app=pn.Row(controls,main,sizing_mode="stretch_width"); app.servable(title="spharmgrid — ERA5 850-hPa spherical harmonic explorer"); app